In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 量子ビットを測定する

<details>
<summary><b>Package versions</b></summary>

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することをお勧めします。

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

量子ビットの状態に関する情報を取得するには、[古典ビット](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Clbit)に_測定_することができます。Qiskitでは、測定は計算基底、すなわち単一量子ビットのPauli-$Z$基底で実行されます。したがって、測定はPauli-$Z$固有状態$|0\rangle$および$|1\rangle$との重なりに応じて、0または1を返します。

$$
|q\rangle \xrightarrow{measure}\begin{cases}
      0 (\text{outcome}+1), \text{with probability } p_0=|\langle q|0\rangle|^{2}\text{,} \\
      1 (\text{outcome}-1), \text{with probability } p_1=|\langle q|1\rangle|^{2}\text{.}
    \end{cases}
$$

## 回路中測定 {#Mid-circuit-measurements}

回路中測定（Mid-circuit measurements）は動的回路の重要な構成要素です。`qiskit-ibm-runtime` v0.43.0より前のバージョンでは、`measure`がQiskitにおける唯一の測定命令でした。しかし、回路中測定は_終端_測定（回路の末尾で行われる測定）とは異なるチューニング要件があります。たとえば、回路中測定をチューニングする際には命令の継続時間を考慮する必要があります。これは、継続時間が長い命令ほどノイズの多い回路になるためです。一方、終端測定の後には命令が存在しないため、終端測定では命令の継続時間を考慮する必要はありません。

`qiskit-ibm-runtime` v0.43.0において、`MidCircuitMeasure`命令が導入されました。名前が示す通り、これはIBM&reg; QPU上での回路中用に最適化された新しい測定命令です。

> **Note:** `MidCircuitMeasure`命令は、バックエンドの`supported_instructions`に報告される`measure_2`命令にマッピングされます。ただし、`measure_2`はすべてのバックエンドでサポートされているわけではありません。`service.backends(filters=lambda b: "measure_2" in b.supported_instructions)`を使用して、対応するバックエンドを見つけることができます。将来的に新しい測定が追加される可能性がありますが、これは保証されていません。

## 回路に測定を適用する {#Apply-a-measurement-to-a-circuit}

回路に測定を適用する方法はいくつかあります。

### `QuantumCircuit.measure`メソッド {#QuantumCircuit-measure-method}

[`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class)を測定するには、[`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure)メソッドを使用します。

例：

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(5, 5)
qc.x(0)
qc.x(1)
qc.x(4)
qc.measure(
    range(5), range(5)
)  #  Measures all qubits into the corresponding clbit.

In [2]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure(1, 0)  # Measure qubit 1 into the classical bit 0.

### `Measure` class

The Qiskit [Measure](/docs/api/qiskit/circuit#qiskit.circuit.Measure) class measures the specified qubits.

In [3]:
from qiskit.circuit import Measure

qc = QuantumCircuit(3, 1)
qc.x([0, 1])
qc.append(Measure(), [0], [0])  # measure qubit 0 into clbit 0

### `QuantumCircuit.measure_all` method

To measure all qubits into the corresponding classical bits, use the [`measure_all`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) method. By default, this method adds new classical bits in a `ClassicalRegister` to store these measurements.

In [4]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_all()  # Measure all qubits.

### `Measure`クラス {#Measure-class}

Qiskitの[Measure](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Measure)クラスは、指定された量子ビットを測定します。

In [5]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_active()  # Measure qubits that are not idle, that is, qubits 0 and 2.

<span id="midcircuit"></span>
### `MidCircuitMeasure` method


Use `MidCircuitMeasure` to apply a mid-circuit measurement (requires `qiskit-ibm-runtime` v0.43.0 or later).  While you can use `QuantumCircuit.measure` for a mid-circuit measurement, because of its design, `MidCircuitMeasure` is typically a better choice.  For example, it adds less overhead to your circuit than when using `QuantumCircuit.measure`.

In [6]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.circuit import MidCircuitMeasure
from qiskit.circuit import Measure

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circ = QuantumCircuit(2, 2)
circ.x(0)
circ.append(MidCircuitMeasure(), [0], [0])
# circ.measure([0], [0])
# circ.measure_all()
print(circ.draw(cregbundle=False))

     ┌───┐┌────────────┐
q_0: ┤ X ├┤0           ├
     └───┘│            │
q_1: ─────┤  Measure_2 ├
          │            │
c_0: ═════╡0           ╞
          └────────────┘
c_1: ═══════════════════
                        


### `QuantumCircuit.measure_all`メソッド {#QuantumCircuit-measure_all-method}

すべての量子ビットを対応する古典ビットに測定するには、[`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all)メソッドを使用します。デフォルトでは、このメソッドはこれらの測定結果を格納するための新しい古典ビットを`ClassicalRegister`に追加します。